# Getting Started with eXist-db Notebook

Welcome to **eXist-db Notebook** — a Jupyter-inspired environment for writing and running XQuery interactively.

This notebook introduces the app's key features:

- **Markdown cells** for documentation and narrative
- **Code cells** for live XQuery execution
- **Named cell chaining** — give cells names, and their results become `$name` variables in later cells
- **Result caching** — named cell results are cached server-side, so subsequent cells don't re-evaluate prior queries
- **Rich output** — XML, JSON, HTML, and plain text rendering with syntax highlighting
- **Serialization options** — control how results are displayed
- **Inline linting** — errors are caught as you type

Run each cell with **Shift+Enter**, or use the toolbar buttons.

## Your First XQuery Expression

XQuery is a powerful language for querying and transforming data. Let’s start simple:

In [ ]:
(: A simple arithmetic expression :)
2 + 2

The default **adaptive** serialization shows results as plain text. XQuery can also work with strings:

In [ ]:
let $name := "World"
return concat("Hello, ", $name, "!")

## Generating XML

XQuery’s native strength is XML. Select **xml** from the serialization dropdown (in the cell’s action bar) to see formatted XML output.

In [ ]:
(:~
 : Generate an XML document
 :
 : @output method=xml indent=yes
 :)
<catalog>
    {
        for $item in ("XQuery", "XSLT", "XPath", "XSD", "XProc")
        return
            <technology type="xml-family">
                <name>{$item}</name>
                <length>{string-length($item)}</length>
            </technology>
    }
</catalog>

## Working with JSON

XQuery 3.1 and 4.0 have excellent support for maps and arrays, which serialize directly to JSON. Select **json** serialization to see the output as formatted JSON.

In [ ]:
(:~
 : Build a JSON structure using XQuery maps and arrays
 :
 : @output method=json
 :)
map {
    "title": "XQuery Notebook",
    "version": 1.0,
    "features": array {
        "live execution",
        "cell chaining",
        "syntax highlighting",
        "rich output"
    },
    "metadata": map {
        "author": "eXist-db",
        "cells": 42,
        "active": true()
    }
}

## HTML Output

Select **html** serialization to render HTML directly in the output area. This is great for creating rich visualizations.

In [ ]:
(:~
 : Generate an HTML table
 :
 : @output method=html media-type=text/html
 :)
<div>
    <h3 style="color: #2563eb; margin-bottom: 0.5em;">Fibonacci Sequence</h3>
    <table style="border-collapse: collapse; width: 100%;">
        <thead>
            <tr style="background: #f0f4ff;">
                <th style="border: 1px solid #ddd; padding: 8px;">n</th>
                <th style="border: 1px solid #ddd; padding: 8px;">F(n)</th>
            </tr>
        </thead>
        <tbody>{
            let $fib := function($self, $n) {
                if ($n le 1) then $n
                else $self($self, $n - 1) + $self($self, $n - 2)
            }
            for $n in 0 to 10
            return
                <tr>
                    <td style="border: 1px solid #ddd; padding: 8px; text-align: center;">{$n}</td>
                    <td style="border: 1px solid #ddd; padding: 8px; text-align: right; font-weight: bold;">{$fib($fib, $n)}</td>
                </tr>
        }</tbody>
    </table>
</div>

## Named Cell Chaining

One of the most powerful features is **named cell chaining**. Add a `@name` directive in an xqdoc comment at the top of a cell to name it. When you run the cell, two things happen:

1. The result is **cached** server-side (no re-evaluation needed)
2. The result becomes available as **`$name`** in all subsequent cells

This is like defining variables in a Jupyter notebook — but explicit, stable across reordering, and efficient.

The xqdoc `@name` directive works identically in both the Notebook web app and VS Code with the Jupyter kernel.

In [ ]:
(:~
 : XML book catalog for cell chaining examples.
 :
 : @name books
 :)
<books>
    <book year="1999"><title>The Art of XQuery</title><pages>320</pages></book>
    <book year="2004"><title>XQuery from the Experts</title><pages>480</pages></book>
    <book year="2007"><title>XQuery: Search Across a Variety of XML Data</title><pages>544</pages></book>
    <book year="2015"><title>eXist: A NoSQL Document Database</title><pages>388</pages></book>
</books>

In [ ]:
(: This cell references $books from the named cell above :)
(: The result was cached — no re-evaluation needed! :)
for $book in $books//book
where xs:integer($book/@year) >= 2004
order by $book/@year
return
    $book/title/string() || " (" || $book/@year || ", " || $book/pages || " pages)"

In [ ]:
(:~
 : Summarize $books as JSON — XPath works on cached XML nodes
 :
 : @output method=json
 :)
map {
    "totalBooks": count($books//book),
    "totalPages": sum($books//book/pages/xs:integer(.)),
    "averagePages": round(avg($books//book/pages/xs:integer(.))),
    "earliestYear": min($books//book/@year/string()),
    "latestYear": max($books//book/@year/string())
}

## Data Cells

Besides XQuery code cells and Markdown cells, notebooks support **data cells** — inline datasets in XML, JSON, or plain text. Use the `@data` directive to mark a code cell as data:

- `@data xml` — XML documents (parsed as XML nodes)
- `@data json` — JSON objects and arrays (parsed via `parse-json()`)
- `@data text` — plain text (stored as a string)

Combine with `@name` to make the data available as a variable, and `@silent` to suppress the output since it's just data loading.

These directives work identically in the Notebook web app and VS Code.

In [ ]:
(:~
 : Sample personnel data.
 :
 : @name people
 : @data xml
 : @silent
 :)
<people>
    <person age="30">
        <name>Alice</name>
        <role>Engineer</role>
        <city>Portland</city>
    </person>
    <person age="25">
        <name>Bob</name>
        <role>Designer</role>
        <city>Seattle</city>
    </person>
    <person age="35">
        <name>Carol</name>
        <role>Manager</role>
        <city>Portland</city>
    </person>
    <person age="28">
        <name>Dave</name>
        <role>Engineer</role>
        <city>San Francisco</city>
    </person>
</people>

In [ ]:
(: Query the XML data cell — $people is parsed and cached automatically :)
for $person in $people//person[@age > 27]
order by $person/name
return
    $person/name/string() || " (" || $person/role || ", " || $person/city || ")"

In [ ]:
(:~
 : Application configuration.
 :
 : @name config
 : @data json
 : @silent
 :)
{
    "appName": "My Dashboard",
    "version": "2.1",
    "features": ["search", "export", "sharing"],
    "limits": {
        "maxUsers": 100,
        "maxStorage": "10GB"
    }
}

In [ ]:
(: Query the JSON data cell — $config is parsed as a map :)
"App: " || $config?appName || " v" || $config?version
    || " (" || string-join($config?features?*, ", ") || ")"

## Sequences and Iteration

XQuery excels at working with sequences. The **adaptive** serialization shows each item on its own line.

In [ ]:
(: Generate a sequence with FLWOR expressions :)
for $n in 1 to 10
let $square := $n * $n
let $cube := $n * $n * $n
return
    $n || "² = " || $square || ", " || $n || "³ = " || $cube

## Higher-Order Functions

XQuery supports higher-order functions — functions that take other functions as arguments or return them.

In [ ]:
(: Higher-order functions: map, filter, fold :)
let $numbers := 1 to 10
let $doubled := for-each($numbers, function($n) { $n * 2 })
let $evens := filter($numbers, function($n) { $n mod 2 eq 0 })
let $sum := fold-left($numbers, 0, function($acc, $n) { $acc + $n })
return map {
    "original": array { $numbers },
    "doubled": array { $doubled },
    "evens": array { $evens },
    "sum": $sum
} => serialize(map { "method": "json", "indent": true() })

## String Processing

XQuery has a rich set of string functions, including full regex support.

In [ ]:
(: String manipulation :)
let $text := "The quick brown fox jumps over the lazy dog"
return
    string-join((
        "Original: " || $text,
        "Upper: " || upper-case($text),
        "Words: " || count(tokenize($text, "\s+")),
        "Reversed words: " || string-join(reverse(tokenize($text, "\s+")), " "),
        "Contains 'fox': " || contains($text, "fox"),
        "Replace: " || replace($text, "(\w+)", "[$1]")
    ), codepoints-to-string(10))

## Date and Time

XQuery has built-in support for dates, times, and durations.

In [ ]:
(: Date and time operations :)
let $now := current-dateTime()
let $epoch := xs:dateTime("1970-01-01T00:00:00Z")
return
    string-join((
        "Current time: " || format-dateTime($now, "[FNn], [MNn] [D], [Y] at [H01]:[m01]:[s01]"),
        "Day of week: " || format-dateTime($now, "[FNn]"),
        "Days since epoch: " || days-from-duration($now - $epoch),
        "ISO week: " || format-dateTime($now, "[Y]-W[W01]")
    ), codepoints-to-string(10))

## Querying the Database

Since this notebook runs on eXist-db, you can query any collection in the database. Here we query the app’s own data collection.

In [ ]:
(: List all collections under /db/apps :)
for $col in xmldb:get-child-collections("/db/apps")
order by $col
return $col

## Error Handling

Both compile-time and runtime errors are caught and displayed. In the Notebook web app, errors appear inline below the cell. In VS Code, errors appear in the Output panel (View → Output → Jupyter).

In [ ]:
(: This will produce a runtime error :)
1 div 0

## XQuery 4.0 Features

eXist-db 7 includes support for many XQuery 4.0 features. To use XQuery 4.0 syntax, you **must** include the version declaration `xquery version "4.0";` at the top of your cell. Without it, eXist defaults to XQuery 3.1, which will reject 4.0-specific syntax like `otherwise`.

Here are a few highlights.

In [ ]:
xquery version "4.0";

(: The `otherwise` operator — a concise null coalescing pattern :)
let $found := ()
let $fallback := "default value"
return $found otherwise $fallback

In [ ]:
(: The fat arrow operator is XQuery 3.1, so no version declaration needed :)
"  Hello, eXist-db!  "
    => normalize-space()
    => upper-case()
    => tokenize("\s+")
    => string-join(" | ")

## CSV Serialization

eXist-db supports `method="csv"` for serializing XQuery data structures as RFC 4180 CSV. Select **CSV** from the serialization dropdown to render results as a styled table, or **CSV (source)** to see the raw CSV text.

CSV serialization accepts arrays of arrays, sequences of maps, or XML in record/field structure.

In [ ]:
(: Array of arrays — select CSV serialization to see as a table :)
[
    ["Name", "Email", "Score"],
    ["Alice", "alice@example.com", 95],
    ["Bob", "bob@example.com", 87],
    ["Charlie", "charlie@example.com", 92]
]

In [ ]:
(: Sequence of maps with csv.header — select CSV serialization :)
serialize((
    map { "name": "Alice",   "dept": "Engineering", "salary": 95000 },
    map { "name": "Bob",     "dept": "Marketing",   "salary": 82000 },
    map { "name": "Charlie", "dept": "Engineering", "salary": 91000 }
), map {
    "method": "csv",
    "csv.header": true()
})

## Keyboard Shortcuts

| Shortcut | Action |
| --- | --- |
| **Cmd/Ctrl+Enter** | Run current cell (stay in place) |
| **Shift+Enter** | Run current cell and advance to next |
| **Cmd/Ctrl+S** | Save notebook |
| **Cmd/Ctrl+Z** | Undo last cell operation (delete, move) |
| **Escape** | Exit markdown editing mode |
| **Double-click** | Edit a markdown cell |

Text-level undo within a cell is handled by the code editor (also Cmd/Ctrl+Z when the cursor is inside a cell). Notebook-level undo restores deleted or moved cells.

---

*Enjoy exploring XQuery with eXist-db Notebook!*